In [2]:
#!/usr/bin/env python3
import cv2
import csv
import glob
import time
import imutils
import numpy as np
import mediapipe as mp
import mp_utils as mu
from matplotlib import pyplot as plt
from imutils.object_detection import non_max_suppression

mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_face_mesh = mp.solutions.face_mesh
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

In [8]:
import cv2
import mediapipe as mp
import numpy as np

# MediaPipe Pose 설정
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# 카메라 연결
cap = cv2.VideoCapture(0)

# 실시간 포즈 검출 및 세그멘테이션 활성화
with mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=True,  # 배경 분리를 위한 세그멘테이션 옵션
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as pose:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("카메라 프레임을 읽을 수 없습니다.")
            break

        # BGR -> RGB 변환
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False

        # 포즈 및 세그멘테이션 처리
        results = pose.process(frame_rgb)

        # 다시 쓰기 가능 설정 및 BGR 변환
        frame_rgb.flags.writeable = True
        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        # 배경 블러 처리 (원하는 강도로 커널 크기 조절 가능 ex: 21, 21)
        blurred_bg = cv2.GaussianBlur(frame_bgr, (21, 21), 0)

        # 세그멘테이션 마스크가 검출된 경우 배경 블러 적용
        if results.segmentation_mask is not None:
            # 마스크 값을 3채널(BGR)로 확장
            condition = (
                np.stack((results.segmentation_mask,) * 3, axis=-1) > 0.1
            )

            # 인물 영역은 원본(frame_bgr), 배경 영역은 블러(blurred_bg) 합성
            skeleton_frame = np.where(condition, frame_bgr, blurred_bg)
        else:
            skeleton_frame = blurred_bg.copy()

        # 인물 위에 스켈레톤 관절 선 그리기
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                skeleton_frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style(),
            )

        # 원본(Origin)과 배경 블러+스켈레톤(Skeleton) 화면 좌우 결합
        combined_view = np.hstack((frame_bgr, skeleton_frame))

        # 텍스트 라벨 추가
        w = frame_bgr.shape[1]
        cv2.putText(
            combined_view,
            "Origin",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2,
        )
        cv2.putText(
            combined_view,
            "Skeleton+blur",
            (w + 20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2,
        )

        # 화면 출력
        cv2.imshow("Origin vs Skeleton with Blur", combined_view)

        # 'q' 키 누르면 종료
        if cv2.waitKey(5) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()